# 4장 응용 — Fashion-MNIST로 신경망 학습하기

4장에서 만든 2층 신경망을 **처음 보는 데이터**에 그대로 써 본다.
Fashion-MNIST는 옷·신발 흑백 사진(28×28)이고, 정답은 10종류다. 파일 형식이 MNIST와 똑같아서 코드는 거의 그대로 쓸 수 있다.

| Part | 내용 |
|---|---|
| 1 | 데이터 살펴보기 — 모양, 종류 이름, 종류별 개수·평균 이미지 |
| 2 | 4장 신경망 그대로 학습하기 |
| 3 | 무엇을 틀리나 — 종류별 정확도, 혼동 행렬, 틀린 사진 |
| 4 | 하이퍼파라미터 실험 — 학습률, 은닉 뉴런 수 |

**표시**: 🔁 배치·행렬 연습 / ⭐ 책 범위 밖 / ⚠️ 함정 / 🔮 실행 전에 예측하기
`# 여기에 작성`을 채우고 확인 셀에서 `통과!`가 나오면 성공. 막히면 `ch04_fashion_solutions.ipynb`.

## 0. 준비
**이 셀부터** 실행하자.
- 채점 도우미 `check`, 그래프 한글 글꼴 설정
- `TwoLayerNet`: 4장 5-2에서 직접 만든 클래스 그대로
- ⭐ `gradient()`: 5장 오차역전파법으로 빠르게 기울기 구하기 (4장 5-3과 같은 코드, 실행만)

In [ ]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
import matplotlib.pyplot as plt
from common.functions import sigmoid, softmax, cross_entropy_error, sigmoid_grad
from common.gradient import numerical_gradient

plt.rcParams['font.family'] = 'Malgun Gothic'   # 그래프 제목에 한글 쓰기 (윈도우 기본 글꼴)
plt.rcParams['axes.unicode_minus'] = False      # 한글 글꼴에서 마이너스 기호 깨짐 방지

def check(got, want, name='답'):
    if got is None:
        raise AssertionError(f'{name}: 아직 안 채웠어요 (None)')
    if isinstance(want, (tuple, str, bool)):
        ok = got == want
    else:
        ok = np.shape(got) == np.shape(want) and np.allclose(got, want)
    mark = '✅' if ok else '❌'
    print(f'{name} = {got!r}   (정답 {want!r}) {mark}')
    assert ok, f'{name} 이(가) 틀렸어요'

class TwoLayerNet:   # 4장 5-2
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        a1 = np.dot(x, self.params['W1']) + self.params['b1']
        z1 = sigmoid(a1)
        a2 = np.dot(z1, self.params['W2']) + self.params['b2']
        return softmax(a2)

    def loss(self, x, t):
        return cross_entropy_error(self.predict(x), t)

    def accuracy(self, x, t):
        y = np.argmax(self.predict(x), axis=1)
        t = np.argmax(t, axis=1)
        return np.sum(y == t) / float(x.shape[0])

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        grads = {}
        for key in ('W1', 'b1', 'W2', 'b2'):
            grads[key] = numerical_gradient(loss_W, self.params[key])
        return grads

    def gradient(self, x, t):   # ⭐ 5장 오차역전파법 — 실행만
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}
        batch_num = x.shape[0]
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)
        dy = (y - t) / batch_num
        grads['W2'] = np.dot(z1.T, dy)
        grads['b2'] = np.sum(dy, axis=0)
        dz1 = sigmoid_grad(a1) * np.dot(dy, W2.T)
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)
        return grads

print('준비 완료')

# Part 1. 데이터 살펴보기
## 1-1. 🔮 불러오기 전에 모양 예측하기
`dataset\fashion_mnist.py`의 `load_fashion_mnist`는 책의 `load_mnist`와 사용법이 똑같다.
`normalize=True, one_hot_label=True`로 불러오면 `x_train.shape`, `t_train.shape`은? (튜플)

In [ ]:
shape_x = None   # 여기에 작성
shape_t = None   # 여기에 작성

In [ ]:
# 불러오기 (실행만) + 확인 — 처음 한 번은 변환하느라 몇 초 걸린다
from dataset.fashion_mnist import load_fashion_mnist, CLASS_NAMES

(x_train, t_train), (x_test, t_test) = load_fashion_mnist(normalize=True, one_hot_label=True)
print(x_train.shape, t_train.shape, x_test.shape, t_test.shape)
print('종류 이름:', CLASS_NAMES)
check(shape_x, (60000, 784), 'shape_x')
check(shape_t, (60000, 10), 'shape_t')
print('통과!')

## 1-2. 정답 숫자 → 옷 이름
`CLASS_NAMES`는 정답 숫자 0~9가 뜻하는 이름 목록이다. (`CLASS_NAMES[9]` = `'앵클부츠'`)
- `labels10`: 훈련 데이터 **앞 10개**의 정답 숫자 (원-핫 → 숫자, `axis` 주의)
- `names10`: 그 10개를 이름으로 바꾼 리스트 — `[CLASS_NAMES[k] for k in labels10]` 같은 **리스트 컴프리헨션** 또는 반복문

In [ ]:
labels10 = None   # 여기에 작성
names10 = None    # 여기에 작성

In [ ]:
# 확인
print(labels10, names10)
check(labels10, np.array([9, 0, 0, 3, 0, 2, 7, 2, 5, 5]), 'labels10')
check(names10[0], '앵클부츠', 'names10[0]')
check(len(names10), 10, 'len(names10)')
print('통과!')

## 1-3. 사진 10장 한꺼번에 보기
3장 6-2에서는 `plt.imshow`로 한 장만 봤다. 여러 장은 `plt.subplot(행 수, 열 수, 번호)`로 칸을 나눠 그린다. (번호는 **1부터**)

반복문 안의 두 줄을 채우자.
- `img`: `i`번째 훈련 이미지를 `(28, 28)`로 되돌린 것 (`reshape`)
- `name`: 그 이미지의 옷 이름 (1-2의 `names10` 활용)

In [ ]:
plt.figure(figsize=(10, 4.5))
for i in range(10):
    img = None    # 여기에 작성
    name = None   # 여기에 작성

    plt.subplot(2, 5, i + 1)   # 2행 5열 중 (i+1)번째 칸
    plt.imshow(img, cmap='gray')
    plt.title(name)
    plt.axis('off')            # 눈금 숨기기
plt.show()

In [ ]:
# 확인
check(img.shape, (28, 28), 'img.shape (마지막 사진)')
check(name, names10[9], 'name (마지막 사진)')
print('통과!')

## 1-4. 🔁 종류별 개수 — 한 줄로
원-핫 정답 `t_train` (60000, 10)에서 **각 열**은 한 종류다. 열마다 1이 몇 개인지 세면 종류별 개수가 된다.
- 🔮 `predict_counts_shape`: 결과 모양은? (튜플)
- `counts`: 종류별 훈련 데이터 개수 (반복문 없이 `np.sum`과 `axis` 한 번)

In [ ]:
predict_counts_shape = None   # 여기에 작성
counts = None                 # 여기에 작성

In [ ]:
# 확인
print(counts)
check(predict_counts_shape, (10,), 'predict_counts_shape')
check(counts, np.full(10, 6000), 'counts')
print('통과! 종류마다 6000장씩 고르게 들어 있다.')

## 1-5. 🔁 종류별 "평균 사진"
같은 종류 사진 6000장을 픽셀마다 평균 내면 그 종류의 대표 모양이 보인다.

**새 도구: 불리언 인덱싱** — `True/False` 배열로 행을 고를 수 있다. (`True`인 행만 남는다)
```python
a = np.array([10, 20, 30, 40])
a[np.array([True, False, True, False])]   # [10, 30]
a[a > 15]                                 # [20, 30, 40]
```

1. `labels_all`: 6만 개 정답 숫자 (60000,)
2. `mean_imgs`: (10, 784) — `c`번째 행 = 종류 `c` 사진들의 평균
   - `x_train[labels_all == c]`로 종류 `c`의 사진만 고르고 → `axis=0`으로 평균 (`np.mean`)

In [ ]:
labels_all = None   # 여기에 작성

mean_imgs = np.zeros((10, 784))
for c in range(10):
    pass   # 여기에 작성

In [ ]:
# 확인 — ⭐ 같은 결과를 행렬 곱 한 번으로도 구할 수 있다: (10, 60000) · (60000, 784) / 6000
check(labels_all.shape, (60000,), 'labels_all.shape')
check(mean_imgs.shape, (10, 784), 'mean_imgs.shape')
check(bool(np.abs(mean_imgs - (t_train.T @ x_train) / 6000).max() < 1e-5), True, 'mean_imgs 값 (행렬 곱 결과와 비교)')

plt.figure(figsize=(10, 4.5))
for c in range(10):
    plt.subplot(2, 5, c + 1)
    plt.imshow(mean_imgs[c].reshape(28, 28), cmap='gray')
    plt.title(CLASS_NAMES[c])
    plt.axis('off')
plt.show()
print('통과! 티셔츠·풀오버·코트·셔츠의 평균 모양이 비슷하다 → 헷갈리기 쉬울 것 같다')

# Part 2. 4장 신경망 그대로 학습하기
4장 5-5와 같은 설정이다. (입력 784, 은닉 50, 출력 10, 10000번, 배치 100, 학습률 0.1)
반복문의 빈칸을 채우자. 20~30초쯤 걸린다.

### 🔮 먼저 예측하기
MNIST는 같은 설정으로 시험 정확도가 약 **94.7%**였다.
- `predict_compare`: Fashion-MNIST는 MNIST보다 `'높다'` / `'비슷하다'` / `'낮다'`?

In [ ]:
predict_compare = None   # 여기에 작성

In [ ]:
np.random.seed(0)
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1
iter_per_epoch = max(train_size / batch_size, 1)

train_loss_list = []
train_acc_list = []
test_acc_list = []

for i in range(iters_num):
    # 1. 미니배치
    batch_mask = None   # 여기에 작성
    x_batch = None      # 여기에 작성
    t_batch = None      # 여기에 작성

    # 2. 기울기 (⭐ gradient)
    grad = None   # 여기에 작성

    # 3. 갱신 (키 4개)
    # 여기에 작성

    train_loss_list.append(network.loss(x_batch, t_batch))

    # 4. 에폭마다 정확도
    if i % iter_per_epoch == 0:
        train_acc_list.append(network.accuracy(x_train, t_train))
        test_acc_list.append(network.accuracy(x_test, t_test))
        print(f'train acc {train_acc_list[-1]:.4f} | test acc {test_acc_list[-1]:.4f}')

In [ ]:
# 확인 + 그래프 (손실 / 정확도)
final_test = test_acc_list[-1]
print('마지막 시험 정확도:', final_test, ' (MNIST는 약 0.947)')
check(len(train_acc_list), 17, 'len(train_acc_list)')
check(bool(final_test > 0.80), True, '시험 정확도 > 80%')
check(predict_compare, '낮다' if final_test < 0.94 else '비슷하다', 'predict_compare')

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(train_loss_list, linewidth=0.5)
plt.title('손실'); plt.xlabel('반복'); plt.ylabel('loss')
plt.subplot(1, 2, 2)
plt.plot(train_acc_list, label='train')
plt.plot(test_acc_list, '--', label='test')
plt.title('정확도'); plt.xlabel('에폭'); plt.ylim(0, 1); plt.legend()
plt.show()
print('통과!')

# Part 3. 무엇을 틀리나
## 3-1. 🔁 시험 데이터 1만 장을 한 번에 예측
- `pred`: 시험 이미지 1만 장의 예측 숫자 (10000,) — `network.predict` 한 번 + `argmax`
- `t_label`: 시험 정답 숫자 (10000,)
- `acc_all`: 정확도 (`pred`와 `t_label` 비교)

In [ ]:
pred = None      # 여기에 작성
t_label = None   # 여기에 작성
acc_all = None   # 여기에 작성

In [ ]:
# 확인
check(pred.shape, (10000,), 'pred.shape')
check(acc_all, network.accuracy(x_test, t_test), 'acc_all (network.accuracy와 같아야 함)')
print('참고: Part 2의 마지막 기록은 9600번째 반복 때 값이라 지금(10000번 뒤)과 조금 다를 수 있다:', test_acc_list[-1])
print('통과!')

## 3-2. 🔁 종류별 정확도
종류마다 "그 종류 사진 중 몇 %를 맞혔나"를 구하자.
- `class_acc`: (10,) — `c`번째 = 정답이 `c`인 사진들 중 맞힌 비율
  - `mask = (t_label == c)` → `pred[mask]`가 `c`인 비율 (`np.mean(... == c)`: True를 1로 평균 = 비율)
- 막대그래프로 그리기: `plt.barh(CLASS_NAMES, class_acc)` (가로 막대)
- `worst`: 가장 못 맞힌 종류의 **이름**

In [ ]:
class_acc = np.zeros(10)
for c in range(10):
    pass   # 여기에 작성

# 그래프: 여기에 작성

worst = None   # 여기에 작성

In [ ]:
# 확인
print(np.round(class_acc, 3))
check(class_acc.shape, (10,), 'class_acc.shape')
check(np.mean(class_acc), acc_all, 'class_acc 평균 (종류마다 1000장이라 전체 정확도와 같다)')
check(worst, CLASS_NAMES[int(np.argmin(class_acc))], 'worst')
print('통과!')

## 3-3. 혼동 행렬 — 무엇을 무엇으로 착각했나
`cm[i, j]` = **정답이 `i`인데 `j`로 예측한** 사진 수 (10, 10). 대각선(`i == j`)이 맞힌 것이다.

### ⚠️🔮 먼저 예측하기 — 짝지어 꺼내기로 한 번에 세면?
4장에서 배운 `y[행 목록, 열 목록]`을 쓰면 한 줄로 될 것 같다.
```python
cm_wrong = np.zeros((10, 10))
cm_wrong[t_label, pred] += 1
```
- `predict_sum`: `cm_wrong.sum()`이 1만(사진 수)이 될까? `True` / `False`

In [ ]:
predict_sum = None   # 여기에 작성

In [ ]:
# 확인 (이제 실행해서 맞춰 보기)
cm_wrong = np.zeros((10, 10))
cm_wrong[t_label, pred] += 1
print('cm_wrong.sum() =', cm_wrong.sum(), '  (사진은 10000장)')
check(predict_sum, bool(cm_wrong.sum() == 10000), 'predict_sum')
print('통과!')

**왜?** `cm_wrong[t_label, pred] += 1`은 `cm_wrong[t_label, pred] = cm_wrong[t_label, pred] + 1`과 같다.
오른쪽에서 **원래 값(0)을 한꺼번에 읽고** 1을 더한 뒤 쓴다. 같은 칸이 1000번 나와도 "0 + 1"을 1000번 **덮어쓸** 뿐이라 1이 된다.
(그래서 최대 100칸 × 1 = 100 이하)

이제 올바르게 세자.
- `cm`: 반복문 `for a, b in zip(t_label, pred):`로 한 장씩 `cm[a, b] += 1`
  (`zip`: 두 목록을 같은 순서끼리 짝지어 꺼내기)
- ⭐ 한 줄 버전: `np.add.at(cm, (t_label, pred), 1)` — 같은 칸이 여러 번 나와도 **모두** 더해 준다 (책 범위 밖)

In [ ]:
cm = np.zeros((10, 10), dtype=int)
# 여기에 작성

In [ ]:
# 확인 + 그림
check(cm.sum(), 10000, 'cm.sum()')
check(cm.sum(axis=1), np.full(10, 1000), '행 합 (종류마다 1000장)')
check(np.trace(cm), np.sum(pred == t_label), '대각선 합 = 맞힌 수')   # np.trace: 대각선 합

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')   # ⭐ 행렬을 색으로 그리기
plt.colorbar()
plt.xticks(range(10), CLASS_NAMES, rotation=60)
plt.yticks(range(10), CLASS_NAMES)
plt.xlabel('예측'); plt.ylabel('정답')
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.show()
print('통과!')

### 생각해 보기
- `confused_with`: 정답이 **셔츠**(6번)인 사진을 셔츠가 **아닌** 것 중 가장 많이 무엇으로 착각했나? (이름)
  - `cm`의 6번 행에서 대각선(6번 칸)을 뺀 최댓값의 위치. 행을 `.copy()`해서 6번 칸을 0으로 바꾼 뒤 `argmax`
  - ⚠️ `.copy()` 없이 고치면 `cm` 자체가 바뀐다 (이름표와 상자!)

In [ ]:
row6 = None            # 여기에 작성
confused_with = None   # 여기에 작성

In [ ]:
# 확인
_r = np.array(cm[6]); _r[6] = 0
check(confused_with, CLASS_NAMES[int(np.argmax(_r))], 'confused_with')
check(int(cm[6, 6]) > 0, True, 'cm[6, 6]이 그대로인가 (copy 했나)')
print('통과!')

## 3-4. 틀린 사진 보기
- `wrong_idx`: 틀린 시험 사진의 **번호** 목록 — `np.arange(10000)`을 불리언 인덱싱으로 거르기 (`pred != t_label`)
- 앞 10장을 `정답 → 예측` 제목으로 그리기 (1-3과 같은 방식)

In [ ]:
wrong_idx = None   # 여기에 작성

# 그래프: 여기에 작성

In [ ]:
# 확인
check(len(wrong_idx), int(np.sum(pred != t_label)), 'len(wrong_idx)')
check(bool(np.all(pred[wrong_idx] != t_label[wrong_idx])), True, '전부 틀린 사진인가')
print('통과! 사람이 봐도 헷갈리는 사진이 꽤 있다.')

# Part 4. 🔁 하이퍼파라미터 실험
학습률, 은닉 뉴런 수 같은 값은 **사람이 정하는** 하이퍼파라미터다. 바꿔 가며 결과를 비교해 보자.
실험을 여러 번 하려면 학습 과정을 **함수**로 묶는 게 편하다.

## 4-1. 학습을 함수로 묶기
`run(lr, hidden_size, iters_num=2000)`
- `np.random.seed(0)`으로 시작 (실험끼리 공평하게 같은 난수)
- Part 2의 반복문에서 **에폭 정확도 부분만 빼고** 그대로
- 돌려주는 것: `(손실 리스트, 마지막 시험 정확도)` — 튜플로 두 개를 한 번에 `return a, b`

> 결과를 빨리 보려고 2000번만 돌린다. (3~5초)

In [ ]:
def run(lr, hidden_size, iters_num=2000):
    np.random.seed(0)
    net = TwoLayerNet(input_size=784, hidden_size=hidden_size, output_size=10)
    losses = []
    # 여기에 작성 (반복문)

    test_acc = net.accuracy(x_test, t_test)
    return losses, test_acc

In [ ]:
# 확인: 아주 짧게 돌려 보기
l_try, a_try = run(0.1, 50, iters_num=50)
check(len(l_try), 50, 'len(losses)')
check(bool(0 < a_try < 1), True, '0 < test_acc < 1')
print('통과!')

## 4-2. 학습률 비교 — 손실 기록을 행렬로
학습률 `[0.01, 0.1, 0.5]` 세 가지로 `run`을 돌려서
- `loss_mat`: 손실 기록 3개를 **행으로 쌓은 행렬** — 행 하나 = 학습률 하나
- `acc_lr`: 시험 정확도 3개 (리스트 또는 배열)

### 🔮 먼저 예측하기
- `predict_loss_mat`: `loss_mat.shape`은? (튜플)

In [ ]:
predict_loss_mat = None   # 여기에 작성

In [ ]:
lrs = [0.01, 0.1, 0.5]

loss_mat = None   # 여기에 작성
acc_lr = None     # 여기에 작성

In [ ]:
# 확인 + 그래프: 행 하나씩 꺼내서 한 그림에
check(predict_loss_mat, (3, 2000), 'predict_loss_mat')
check(loss_mat.shape, (3, 2000), 'loss_mat.shape')

plt.figure(figsize=(8, 4))
for lr, row, acc in zip(lrs, loss_mat, acc_lr):
    plt.plot(row, linewidth=0.6, label=f'lr={lr}  (test acc {acc:.3f})')
plt.ylim(0, 2.5); plt.xlabel('반복'); plt.ylabel('loss'); plt.legend()
plt.show()
print('통과!')

### 생각해 보기
- `best_lr`: 2000번 안에서 **마지막 100번의 평균 손실**이 가장 작은 학습률은? (숫자) — 그래프를 보고 적거나, `loss_mat[:, -100:]`의 행마다 평균 → `argmin`

In [ ]:
best_lr = None   # 여기에 작성

In [ ]:
# 확인
_means = np.mean(loss_mat[:, -100:], axis=1)
print('마지막 100번 평균 손실:', dict(zip(lrs, np.round(_means, 3))))
check(best_lr, lrs[int(np.argmin(_means))], 'best_lr')
print('통과! 2000번처럼 짧게 돌릴 때는 큰 학습률이 빨리 내려간다. 너무 크면 4장 4-5처럼 발산할 수 있다.')

## 4-3. 은닉 뉴런 수 비교
학습률 0.1로 은닉 뉴런 `[10, 50, 200]`을 비교하자.
- `acc_hidden`: 시험 정확도 3개
- 막대그래프: `plt.bar([str(h) for h in hiddens], acc_hidden)`

In [ ]:
hiddens = [10, 50, 200]

acc_hidden = None   # 여기에 작성

# 그래프: 여기에 작성

In [ ]:
# 확인
print(dict(zip(hiddens, np.round(acc_hidden, 4))))
check(len(acc_hidden), 3, 'len(acc_hidden)')
check(bool(all(0.5 < a < 1 for a in acc_hidden)), True, '정확도 범위')
print('통과!')

# 마무리 — 생각해 보기 (답 없음, 노트에 적어 보기)
1. MNIST보다 정확도가 낮은 이유를 1-5(평균 사진)와 3-3(혼동 행렬)을 근거로 설명해 보자.
2. Part 2 그래프에서 훈련 정확도와 시험 정확도의 차이는? 오버피팅이 있었나?
3. 은닉 뉴런을 늘리면 정확도는 올랐나? 계산 시간은?
4. 정확도를 더 올리려면? → 6장(최적화 방법, 가중치 초깃값, 드롭아웃), 7장(CNN: 이미지의 모양을 살려서 학습)에서 배운다.